# Autoencoders, GANs, Diffusion

Autoencoders: learn dense representations of input data: called **latent representations**
- Without any supervision
- Good for anomaly detection, pretraining of LLMs, generating new data (some of them)

General Adversarial Networks (GANs): Generate very convincing fake data
- Super Resolution (increase image res)
- Colorization
- Image gen

Diffusion Models -> have been replacing GANs -> easier to train, slower to run

All of these are unsupervised, learn latent representations, and can generate, but they work very differently
- Autoencoders
    - Learn to copy inputs to outputs 
    - Constraining them helps them learn better ways to copy nontrivially
- GANs
    - 2 neural nets: a generator and a discriminator
        - Generator: generate similar data to training set
        - Discriminator: tell real from fake
    - The generator and discriminator compete turning training (adversarial training)
- Diffusion
    - Gradually remove noise from an image

## PCA w/ Undercomplete Autoencoder

If an autoencoder uses linear activations, MSE cost function, then its just PCA

In [1]:
import torch
import torch.nn as nn

device = "cuda:0"
torch.manual_seed(42)
encoder = nn.Linear(3, 2)
decoder = nn.Linear(2, 3)
autoencoder = nn.Sequential(encoder, decoder).to(device)

This does all the same stuff that PCA in Ch7 did. For example, if you are going 3D->2D, itll find the plane that preserves the most variance

## Stacked Autoencoders
An autoencoder with >1 layer

Obviously more stacking means more powerful, but you also run the risk of overfitting to the training set if it has enough parameters to memorize the training set

Typically, any stacked autoencoder is symmetric. For example, 784 input neurons -> 128 -> 32 -> 128 -> 784. The encodings are the ones AFTER 32

### In Pytorch

In [ ]:
stacked_encoder = nn.Sequential(
    nn.Flatten(),
    nn.Linear(1 * 28 * 28, 128), nn.ReLU(),
    nn.Linear(128, 32), nn.ReLU(),
) # this layout is for mnist, but you can change it for whatever dataset
stacked_decoder = nn.Sequential(
    nn.Linear(32, 128), nn.ReLU(),
    nn.Linear(128, 1 * 28 * 28), nn.Sigmoid(), # the sigmoid is for outputs -> you want greyscale values btwn 0 and 1 for mnist
    nn.Unflatten(dim=1, unflattened_size=(1, 28, 28))
)
stacked_autoencoder = nn.Sequential(stacked_encoder, stacked_decoder).to(device)

Training this shows very lossy/blurry reconstructions as the output

You could train for longer or make the model bigger

### Anomaly Detection w Autoencoders

Just calculate the reconstruction loss between some image and the output. For example, an industrial process where you make square boxes. A bad box might get ripped in half, and not be in the training set, and the model might reconstruct it as fully intact - thus loss high.

### Visualizing Autoencoder Output

Generally, an autoencoder that goes down to 2 or 3 dimensions wont be useful - but you can take one that goes down to 32-dim and use something like t-SNE to visualize

In [ ]:
from sklearn.manifold import TSNE

# with torch.no_grad():
#     X_valid_compressed = stacked_encoder(X_valid.to(device))

# tsne = TSNE(init="pca", learning_rate="auto", random_state=42)
# X_valid_2D = tsne.fit_transform(X_valid_compressed.cpu())

### Unsupervised Pretraining Using Stacked Autoencoders

Common technique for low-data supervised tasks is to find a similar model and reuse its lower layers -> similar idea w autoencoders

1) train a stacked autoencoder on all your data.
2) reuse the lower layers as the start to a neural net
3) potentially freeze the kept layers when training the new head


### Tying Weights

To prevent overfitting, you tie the weights of the encoder and decoder. Halves the parameter count, speeds up training

In [5]:
import torch.nn.functional as F

class TiedAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc1 = nn.Linear(1 * 28 * 28, 128)
        self.enc2 = nn.Linear(128, 32)
        self.dec1_bias = nn.Parameter(torch.zeros(128))
        self.dec2_bias = nn.Parameter(torch.zeros(1 * 28 * 28))

    def encode(self, X):
        Z = X.view(-1, 1 * 28 * 28)  # flatten
        Z = F.relu(self.enc1(Z))
        return F.relu(self.enc2(Z))

    def decode(self, X):
        Z = F.relu(F.linear(X, self.enc2.weight.t(), self.dec1_bias))
        Z = F.sigmoid(F.linear(Z, self.enc1.weight.t(), self.dec2_bias))
        return Z.view(-1, 1, 28, 28)  # unflatten

    def forward(self, X):
        return self.decode(self.encode(X))

### Training one Autoencoder at a time

'Greedy layerwise training'

Train a shallow autoencoder, then encode your whole dataset. Then train another shallow autoencoder on the encoded dataset, and encode again. Repeat.

Your end model is just all the encoders sandwiched together into one model -> think gradient boosting in tabular ML, underfit a bunch until it gets close

## Convolutional Autoencoders

